# Lesson 8: 方言翻译（LoRA 微调）教程

本教程演示如何使用 LoRA (Low-Rank Adaptation) 技术微调大语言模型，实现方言到普通话的翻译。

## 学习目标
- 理解 LoRA 参数高效微调原理
- 学习大语言模型的加载和量化
- 掌握方言翻译任务的数据准备
- 实现 LoRA 微调流程
- 评估翻译质量

## 1. 环境准备

In [ ]:
import sys
from pathlib import Path

# 添加项目根目录到路径
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

# 检查是否安装了 peft
try:
    from peft import LoraConfig, get_peft_model, TaskType
    print("✓ PEFT 已安装")
except ImportError:
    print("✗ PEFT 未安装")
    print("请运行: pip install peft")

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
print("环境准备完成！")

## 2. LoRA 原理

### 什么是 LoRA？

LoRA (Low-Rank Adaptation) 是一种参数高效的微调方法，通过在预训练模型的权重矩阵旁边添加低秩矩阵来实现微调。

### 核心思想

对于预训练权重矩阵 $W_0 \in \mathbb{R}^{d \times k}$，LoRA 添加一个低秩分解：

$$W = W_0 + \Delta W = W_0 + BA$$

其中：
- $B \in \mathbb{R}^{d \times r}$
- $A \in \mathbb{R}^{r \times k}$
- $r \ll \min(d, k)$ (rank，通常为 8 或 16)

### 优势

1. **参数效率**: 只需训练 $r(d+k)$ 个参数，而不是 $d \times k$ 个
2. **显存节省**: 冻结原始权重，只更新 LoRA 参数
3. **模块化**: 可以为不同任务训练不同的 LoRA 模块
4. **无推理延迟**: 可以将 LoRA 权重合并到原始权重中

### 示例

对于 7B 参数的模型：
- 全量微调: 需要训练 7B 参数
- LoRA (r=8): 只需训练约 4M 参数（0.06%）

## 3. 数据准备

方言翻译数据格式：
```json
[
    {
        "dialect": "我今日好开心啊",
        "mandarin": "我今天很开心"
    },
    ...
]
```

In [ ]:
# 示例数据
sample_data = [
    {"dialect": "我今日好开心啊", "mandarin": "我今天很开心"},
    {"dialect": "你食咗饭未啊", "mandarin": "你吃饭了吗"},
    {"dialect": "呢度好靓啊", "mandarin": "这里很漂亮"},
    {"dialect": "我哋去边度玩", "mandarin": "我们去哪里玩"},
    {"dialect": "听日见", "mandarin": "明天见"}
]

print("示例数据:")
for i, item in enumerate(sample_data, 1):
    print(f"{i}. 方言: {item['dialect']}")
    print(f"   普通话: {item['mandarin']}")
    print()

## 4. 加载基础模型

我们使用较小的模型进行演示（如 Qwen-1.8B）。在实际应用中，可以使用更大的模型。

In [ ]:
# 模型配置
model_name = "Qwen/Qwen-1_8B-Chat"  # 或使用其他模型

print(f"加载模型: {model_name}")
print("注意: 首次加载会下载模型，可能需要较长时间")
print("\n如果模型不存在，这个单元格会失败。")
print("你可以使用本地模型路径或跳过此步骤。")

# 尝试加载模型（如果存在）
try:
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )
    
    # 使用 8-bit 量化以节省显存
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map='auto',
        trust_remote_code=True,
        load_in_8bit=True  # 8-bit 量化
    )
    
    print("✓ 模型加载成功！")
    print(f"模型参数量: {model.num_parameters() / 1e9:.2f}B")
    
except Exception as e:
    print(f"✗ 模型加载失败: {e}")
    print("\n这是正常的，因为模型可能未下载。")
    print("在实际使用中，请先下载模型或使用本地路径。")

## 5. 配置 LoRA

LoRA 的关键超参数：
- **r (rank)**: 低秩矩阵的秩，通常为 8, 16, 32
- **lora_alpha**: 缩放因子，通常为 r 的 2-4 倍
- **target_modules**: 要应用 LoRA 的模块（如 q_proj, v_proj）
- **lora_dropout**: Dropout 比率

In [ ]:
if 'model' in locals():
    # LoRA 配置
    lora_config = LoraConfig(
        r=8,                              # rank
        lora_alpha=32,                    # alpha = 4 * r
        target_modules=['q_proj', 'v_proj'],  # 目标模块
        lora_dropout=0.1,                 # dropout
        bias='none',                      # 不训练 bias
        task_type=TaskType.CAUSAL_LM      # 任务类型
    )
    
    # 应用 LoRA
    model = get_peft_model(model, lora_config)
    
    # 打印可训练参数
    model.print_trainable_parameters()
    
    print("\n✓ LoRA 配置完成！")
else:
    print("模型未加载，跳过 LoRA 配置")

## 6. 构建训练数据

将方言-普通话对转换为模型输入格式。

In [ ]:
def build_prompt(dialect_text):
    """构建翻译 prompt"""
    return f"""请将以下方言翻译成普通话：
方言：{dialect_text}
普通话："""

def prepare_training_data(data, tokenizer):
    """准备训练数据"""
    inputs = []
    labels = []
    
    for item in data:
        # 构建输入
        prompt = build_prompt(item['dialect'])
        full_text = prompt + item['mandarin']
        
        # Tokenize
        encoding = tokenizer(
            full_text,
            max_length=512,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        
        inputs.append(encoding['input_ids'])
        
        # 创建标签（只计算输出部分的损失）
        prompt_encoding = tokenizer(
            prompt,
            max_length=512,
            truncation=True
        )
        prompt_length = len(prompt_encoding['input_ids'])
        
        label = encoding['input_ids'].clone()
        label[0, :prompt_length] = -100  # 忽略 prompt 部分
        labels.append(label)
    
    return inputs, labels

# 示例
if 'tokenizer' in locals():
    sample_prompt = build_prompt(sample_data[0]['dialect'])
    print("示例 Prompt:")
    print(sample_prompt)
    print(f"\n期望输出: {sample_data[0]['mandarin']}")
else:
    print("Tokenizer 未加载")

## 7. 推理演示（使用预训练模型）

在微调之前，先看看预训练模型的翻译效果。

In [ ]:
def translate(dialect_text, model, tokenizer, max_length=100):
    """翻译方言文本"""
    prompt = build_prompt(dialect_text)
    
    inputs = tokenizer(prompt, return_tensors='pt')
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 提取翻译结果
    if '普通话：' in generated_text:
        translation = generated_text.split('普通话：')[1].strip()
    else:
        translation = generated_text[len(prompt):].strip()
    
    return translation

# 测试翻译
if 'model' in locals() and 'tokenizer' in locals():
    print("测试预训练模型的翻译能力：\n")
    
    for item in sample_data[:3]:
        dialect = item['dialect']
        reference = item['mandarin']
        
        try:
            translation = translate(dialect, model, tokenizer)
            print(f"方言: {dialect}")
            print(f"参考翻译: {reference}")
            print(f"模型翻译: {translation}")
            print()
        except Exception as e:
            print(f"翻译失败: {e}")
            break
else:
    print("模型未加载，跳过推理演示")

## 8. 训练流程（概念演示）

实际训练需要较长时间和大量数据。这里展示训练的基本流程。

In [ ]:
# 训练伪代码（实际训练请使用 scripts/lesson_08_dialect_translation.py）

training_pseudocode = """
# 1. 准备数据
train_data = load_data('train.json')
val_data = load_data('val.json')

# 2. 创建 DataLoader
train_loader = DataLoader(train_data, batch_size=4, shuffle=True)
val_loader = DataLoader(val_data, batch_size=4)

# 3. 配置优化器
optimizer = AdamW(model.parameters(), lr=2e-4)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=100,
    num_training_steps=len(train_loader) * epochs
)

# 4. 训练循环
for epoch in range(epochs):
    model.train()
    for batch in train_loader:
        # 前向传播
        outputs = model(**batch)
        loss = outputs.loss
        
        # 反向传播
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    
    # 验证
    val_loss = evaluate(model, val_loader)
    print(f'Epoch {epoch}: val_loss={val_loss}')

# 5. 保存模型
model.save_pretrained('checkpoints/dialect_translator')
"""

print("训练流程（伪代码）:")
print(training_pseudocode)

print("\n实际训练命令:")
print("python scripts/lesson_08_dialect_translation.py --mode train \\")
print("    --model_name Qwen/Qwen-7B-Chat \\")
print("    --train_data data/dialect_parallel.json \\")
print("    --output_dir checkpoints/dialect_translator \\")
print("    --epochs 3 \\")
print("    --quantization")

## 9. 评估指标

### BLEU Score
衡量机器翻译质量的常用指标，基于 n-gram 匹配。

### 人工评估
- **流畅度**: 翻译是否自然流畅
- **准确度**: 是否保留原意
- **完整度**: 是否遗漏信息

In [ ]:
def simple_bleu(reference, hypothesis):
    """简单的 BLEU-1 计算"""
    ref_words = set(reference)
    hyp_words = set(hypothesis)
    
    matches = len(ref_words & hyp_words)
    total = len(hyp_words)
    
    if total == 0:
        return 0.0
    
    return matches / total

# 示例
reference = "我今天很开心"
hypothesis1 = "我今天很高兴"  # 好的翻译
hypothesis2 = "今天我开心"    # 一般的翻译

score1 = simple_bleu(reference, hypothesis1)
score2 = simple_bleu(reference, hypothesis2)

print(f"参考翻译: {reference}")
print(f"\n假设1: {hypothesis1}")
print(f"BLEU-1: {score1:.2f}")
print(f"\n假设2: {hypothesis2}")
print(f"BLEU-1: {score2:.2f}")

## 10. LoRA 权重合并

训练完成后，可以将 LoRA 权重合并到基础模型中，以便部署。

In [ ]:
# 合并权重（伪代码）
merge_pseudocode = """
from peft import PeftModel

# 1. 加载基础模型
base_model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen-7B-Chat')

# 2. 加载 LoRA 权重
model = PeftModel.from_pretrained(
    base_model,
    'checkpoints/dialect_translator/best'
)

# 3. 合并权重
merged_model = model.merge_and_unload()

# 4. 保存合并后的模型
merged_model.save_pretrained('checkpoints/dialect_translator_merged')

# 现在可以像使用普通模型一样使用合并后的模型
"""

print("LoRA 权重合并（伪代码）:")
print(merge_pseudocode)

## 11. 总结

本教程演示了：
1. ✅ LoRA 参数高效微调原理
2. ✅ 大语言模型的加载和量化
3. ✅ 方言翻译数据准备
4. ✅ LoRA 配置和应用
5. ✅ 训练流程概述
6. ✅ 评估指标

### 关键技术点
- **LoRA**: 只训练 0.1% 的参数，大幅降低显存需求
- **量化**: 使用 8-bit 或 4-bit 量化进一步节省显存
- **Prompt 工程**: 设计合适的 prompt 提高翻译质量
- **参数高效**: 适合资源受限的场景

### 实际训练
要训练自己的模型，运行：
```bash
python scripts/lesson_08_dialect_translation.py --mode train \
    --model_name Qwen/Qwen-7B-Chat \
    --train_data data/dialect_parallel.json \
    --output_dir checkpoints/dialect_translator \
    --epochs 3 \
    --batch_size 4 \
    --lora_r 8 \
    --quantization
```

### 下一步
- 收集更多方言-普通话平行语料
- 尝试不同的 LoRA 配置（r, alpha）
- 使用更大的模型（Qwen-14B, Qwen-72B）
- 实现多方言支持
- 添加上下文理解能力